### Spark session

In [40]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IPL Silver Layer") \
    .getOrCreate()

In [5]:
spark

### Read From Silver


In [11]:
df = spark.read.format('parquet')\
                        .option("inferSchema",True)\
                        .option("headers",True)\
                        .load("../data/silver/ipl_cleaned")

In [5]:
df.show()

+--------+--------------------+--------------------+-----------+---------+----+----+-----------+----------+-----------+---------+--------------------+----+
|match_id|        batting_team|        bowling_team|     batter|   bowler|over|ball|runs_batter|runs_total|is_boundary|is_wicket|               venue|year|
+--------+--------------------+--------------------+-----------+---------+----+----+-----------+----------+-----------+---------+--------------------+----+
|  335982|Kolkata Knight Ri...|Royal Challengers...| SC Ganguly|  P Kumar|   0|   1|          0|         1|          0|        0|M Chinnaswamy Sta...|2008|
|  335982|Kolkata Knight Ri...|Royal Challengers...|BB McCullum|  P Kumar|   0|   2|          0|         0|          0|        0|M Chinnaswamy Sta...|2008|
|  335982|Kolkata Knight Ri...|Royal Challengers...|BB McCullum|  P Kumar|   0|   3|          0|         0|          0|        0|M Chinnaswamy Sta...|2008|
|  335982|Kolkata Knight Ri...|Royal Challengers...|BB McCullum|

In [12]:
df.printSchema()

root
 |-- match_id: integer (nullable = true)
 |-- batting_team: string (nullable = true)
 |-- bowling_team: string (nullable = true)
 |-- batter: string (nullable = true)
 |-- bowler: string (nullable = true)
 |-- over: integer (nullable = true)
 |-- ball: integer (nullable = true)
 |-- runs_batter: integer (nullable = true)
 |-- runs_total: integer (nullable = true)
 |-- is_boundary: integer (nullable = true)
 |-- is_wicket: integer (nullable = true)
 |-- venue: string (nullable = true)
 |-- year: integer (nullable = true)



### Batter Season Performance

In [13]:
batter_stats = df.groupBy(
    "year",
    "batter"
).agg(
    sum("runs_batter").alias("total_runs"),
    count("*").alias("balls"),
    sum("is_boundary").alias("boundaries")
)

In [14]:
batter_stats = batter_stats.withColumn(
    "strike_rate",
    round((col("total_runs") / col("balls")) * 100, 2)
)

In [16]:
window_spec = Window.partitionBy("year").orderBy(desc("total_runs"))

batter_stats = batter_stats.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

In [17]:
batter_stats.show(5, truncate=False, vertical=True)

-RECORD 0--------------------
 year        | 2008          
 batter      | SE Marsh      
 total_runs  | 614           
 balls       | 440           
 boundaries  | 26            
 strike_rate | 139.55        
 rank        | 1             
-RECORD 1--------------------
 year        | 2008          
 batter      | G Gambhir     
 total_runs  | 532           
 balls       | 378           
 boundaries  | 8             
 strike_rate | 140.74        
 rank        | 2             
-RECORD 2--------------------
 year        | 2008          
 batter      | ST Jayasuriya 
 total_runs  | 508           
 balls       | 308           
 boundaries  | 30            
 strike_rate | 164.94        
 rank        | 3             
-RECORD 3--------------------
 year        | 2008          
 batter      | SR Watson     
 total_runs  | 467           
 balls       | 307           
 boundaries  | 19            
 strike_rate | 152.12        
 rank        | 4             
-RECORD 4--------------------
 year     

In [21]:
batter_stats.write.mode("overwrite")\
                           .partitionBy("year")\
                            .parquet("../data/gold/batter_stats")

### Bowler Performance

In [18]:
bowler_stats = df.groupBy(
    "year",
    "bowler"
).agg(
    sum("runs_total").alias("runs_conceded"),
    count("*").alias("balls"),
    sum("is_wicket").alias("wickets")
)

In [19]:
bowler_stats = bowler_stats.withColumn(
    "economy",
    round(col("runs_conceded") / (col("balls") / 6), 2)
)

In [20]:
window_spec2 = Window.partitionBy("year").orderBy(desc("wickets"))

bowler_stats = bowler_stats.withColumn(
    "rank",
    dense_rank().over(window_spec2)
)

In [24]:
bowler_stats.show(5,vertical=True)

-RECORD 0----------------------
 year          | 2008          
 bowler        | Sohail Tanvir 
 runs_conceded | 255           
 balls         | 247           
 wickets       | 24            
 economy       | 6.19          
 rank          | 1             
-RECORD 1----------------------
 year          | 2008          
 bowler        | JA Morkel     
 runs_conceded | 385           
 balls         | 288           
 wickets       | 20            
 economy       | 8.02          
 rank          | 2             
-RECORD 2----------------------
 year          | 2008          
 bowler        | SR Watson     
 runs_conceded | 373           
 balls         | 325           
 wickets       | 20            
 economy       | 6.89          
 rank          | 2             
-RECORD 3----------------------
 year          | 2008          
 bowler        | IK Pathan     
 runs_conceded | 342           
 balls         | 318           
 wickets       | 20            
 economy       | 6.45          
 rank   

In [22]:
bowler_stats.write.mode("overwrite")\
                            .partitionBy("year")\
                            .parquet("../data/gold/bowlers_stats")

### Match Summary

In [25]:
match_summary = df.groupBy(
    "match_id",
    "year",
    "venue"
).agg(
    sum("runs_total").alias("match_total_runs"),
    sum("is_wicket").alias("total_wickets")
)

match_summary.show(10)



+--------+----+--------------------+----------------+-------------+
|match_id|year|               venue|match_total_runs|total_wickets|
+--------+----+--------------------+----------------+-------------+
|  335985|2008|    Wankhede Stadium|             323|           12|
|  419158|2010|        Eden Gardens|             256|           11|
|  335987|2008|Sawai Mansingh St...|             315|           12|
|  335982|2008|M Chinnaswamy Sta...|             284|           13|
|  392239|2009|New Wanderers Sta...|             273|           15|
|  501200|2011|       Nehru Stadium|             298|            9|
|  419141|2010|Vidarbha Cricket ...|             303|           20|
|  392235|2009|     SuperSport Park|             310|           14|
|  392210|2009|           Kingsmead|             320|           12|
|  419112|2010|M Chinnaswamy Sta...|             384|            5|
+--------+----+--------------------+----------------+-------------+
only showing top 10 rows


In [26]:

match_summary.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet("../data/gold/match_summary")

### Venue Stats

In [47]:

venue_stats = df.groupBy("venue", "year") \
    .agg(
        avg("runs_total").alias("avg_runs"),
        countDistinct("match_id").alias("matches")
    )

In [48]:
venue_stats.write.mode("overwrite")\
    .partitionBy("year")\
    .parquet("../data/gold/venue_stats")